Enlace al fork: https://github.com/aleoropgal/bailecomesolo

# **Proyecto de búsqueda: Come Solo**

**Equipo:** Tragón

**Integrantes:**
*   Aníbal Zavala Herrera
*   Jorge Luis Pineda Guzmán
*   Alejandra Oropeza

In [ ]:
import urllib.request
import importlib.util
import numpy as np
import pandas as pd
import time

url = "https://raw.githubusercontent.com/aleoropgal/bailecomesolo/master/src/SimpleSearch.py"

urllib.request.urlretrieve(url, "SimpleSearch.py")

('SimpleSearch.py', <http.client.HTTPMessage at 0x7e122751f390>)

In [ ]:
import SimpleSearch as sp

# **Descripción del Come Solo**



El Come solo (conocido internacionalmente como Peg Solitaire) es un rompecabezas de tablero de estrategia para un solo jugador. En nuestra variante, el juego se desarrolla sobre un tablero triangular compuesto por 15 casillas o huecos dispuestos en 5 filas piramidales.   
Al inicio de la partida, 14 de las 15 casillas se encuentran ocupadas por fichas o pivotes, dejando exactamente 1 casilla vacía. La mecánica básica de juego consiste en realizar saltos: una ficha puede saltar sobre una ficha vecina adyacente que esté inmediatamente al lado, siempre y cuando caiga en una casilla vacía justo detrás de ella. Al efectuar el salto, la ficha sobre la cual se brincó es capturada y eliminada permanentemente del tablero.  
Los saltos válidos pueden ejecutarse en 6 direcciones distintas dentro de la geometría del triángulo:
* Horizontales: Hacia la izquierda o hacia la derecha dentro de la misma fila.
* Diagonales Superiores: Hacia arriba-izquierda o arriba-derecha subiendo de fila.
* Diagonales Inferiores: Hacia abajo-izquierda o abajo-derecha bajando de fila.

El objetivo final del juego es continuar ejecutando saltos válidos de manera consecutiva hasta que quede una sola ficha en todo el tablero. Si en algún punto no existen saltos legales posibles y permanecen 2 o más fichas en el tablero, la partida finaliza en un callejón sin salida o estado de bloqueo (dead-end).   

EJEMPLO:
```
       [0]
     [1] [1]
   [1] [1] [1]
 [1] [1] [1] [1]
[1] [1] [1] [1] [1]
```

En este ejemplo se tiene un espacio vacio en la primera fila.

En base a las reglas del juego se tiene dos posibilidades:

* Opción A: Saltar desde la casilla (2,0) hacia (0,0), pasando sobre la ficha en (1,0).
* Opción B: Saltar desde la casilla (2,2) hacia (0,0), pasando sobre la ficha en (1,1).

Suponiendo que elegimos ejecutar la Opción A: La ficha de la casilla (2,0) se desplaza a (0,0) (pasa de 0 a 1).
La ficha en (2,0) deja su lugar vacío (pasa de 1 a 0).La ficha intermedia en (1,0) es removida del tablero (pasa de 1 a 0).   

El estado resultante tras este movimiento dejaria 13 fichas restantes, y se veria de la siguiente manera:

```
       [1]
     [0] [1]
   [0] [1] [1]
 [1] [1] [1] [1]
[1] [1] [1] [1] [1]

```

Este proceso se repite sucesivamente. Cada acción válida reudce el conteo total de fichas en 1 y modifica la configuracion del tablero




# **Los cuatro componentes**

**1. El estado:** <br>
En nuestro problemas de Come Solo, el estado representa la configuración actual del tablero, esto nos indica qué posiciones contienen una ficha y cuál se encuentra vacía. Para ello, utilizamos una tupla de tuplas que representa las diferentes filas del tablero triangular. Cada posición se representa mediante un 1 cuando tiene una ficha y un 0 cuando está vacía. <br><br>
Se utiliza una tupla de tuplas para que el estado sea hashable. SimpleSearch registra los estados alcanzados mediante un diccionario, y las tuplas permiten que la configuración pueda utilizarse como clave del diccionario. <br>
Además, el estado es mínimo ya que contiene únicamente la información necesaria para describir la configuración del tablero. Los datos como profundidad, padre y el costo, no forman parte de nuestra representación del estado.



In [ ]:
#Definir el estado inicial
start=sp.node(((0,),(1,1),(1,1,1),(1,1,1,1),(1,1,1,1,1)), depth=0, parent=None)

**2. La operación sucesor:** <br>
La función sucesorCS() recibe un nodo y genera una lista con los posibles estados sucesores que pueden alcanzarse.
Para determinar los sucesores, recorremos las posiciones del tablero y buscamos posiciones vacías, cuando encuentra una vacía se comprueba si existen dos fichas consecutivas en alguna de las seis direcciones posibles del tablero:<br><br> *izquierda, derecha, izquierda-superior, derecha-superior, izquierda-inferior  y derecha-inferior.*<br><br>
Siempre volvemos a convertir cada estado nuevo a una tupla de tuplas que almacenamos dentro de un nodo y se establece el nodo actual como padre e incrementamos la profundidad.<br>
Nuestra función genera únicamente movimientos que cumplen con las reglas del Come Solo, evitando hacer movimientos no válidos en el tablero.

In [ ]:
# Definir una funcion sucesor
def sucesorCS(nodo):
    S=[]
    st, n=nodo.state, len(nodo.state)
    for i in range(n):
      for j in range(i+1):
        if st[i][j]==0:

          if j>=2 and st[i][j-1]==1 and st[i][j-2]==1:
            izquierda=[list(row) for row in st] #LISTO
            izquierda[i][j-1]=0
            izquierda[i][j-2]=0
            izquierda[i][j]=1
            S.append(sp.node(tuple(tuple(row) for row in izquierda), depth=nodo.depth+1, parent=nodo))

          if j<=2 and i-2>=j and st[i][j+1]==1 and st[i][j+2]==1:
            derecha=[list(row) for row in st] #LISTO
            derecha[i][j+1]=0
            derecha[i][j+2]=0
            derecha[i][j]=1
            S.append(sp.node(tuple(tuple(row) for row in derecha), depth=nodo.depth+1, parent=nodo))

          if i>=2 and j>=2 and st[i-1][j-1]==1 and st[i-2][j-2]==1:
            izqsup=[list(row) for row in st]
            izqsup[i-1][j-1]=0
            izqsup[i-2][j-2]=0
            izqsup[i][j]=1
            S.append(sp.node(tuple(tuple(row) for row in izqsup), depth=nodo.depth+1, parent=nodo))

          if i>=2 and i-2>=j and st[i-1][j]==1 and st[i-2][j]==1:
            dersup=[list(row) for row in st]
            dersup[i-1][j]=0
            dersup[i-2][j]=0
            dersup[i][j]=1
            S.append(sp.node(tuple(tuple(row) for row in dersup), depth=nodo.depth+1, parent=nodo))

          if i<=n-3 and st[i+1][j]==1 and st[i+2][j]==1:
            izqinf=[list(row) for row in st]
            izqinf[i+1][j]=0
            izqinf[i+2][j]=0
            izqinf[i][j]=1
            S.append(sp.node(tuple(tuple(row) for row in izqinf), depth=nodo.depth+1, parent=nodo))

          if i<=n-3 and j<=2 and st[i+1][j+1]==1 and st[i+2][j+2]==1:
            derinf=[list(row) for row in st]
            derinf[i+1][j+1]=0
            derinf[i+2][j+2]=0
            derinf[i][j]=1
            S.append(sp.node(tuple(tuple(row) for row in derinf), depth=nodo.depth+1, parent=nodo))

    return S

In [ ]:
#sucesorCS(start)

**3. La condición de meta**<br>
La condición de meta para Come Solo es que solamente quede una ficha en el tablero. Para comprobarlo, se utiliza la función contar_fichas(), que recorre todas las posiciones del estado y cuenta las que contienen un 1, entonces la búsqueda termina cuando encuentre un estado con una configuración de exactamente una ficha. <br>

Nuestra función define la solución como una propiedad: tener una sola ficha. <br>
Por lo que no necesitamos comparar contra un estado objetivo específico o conocido.


In [ ]:
# Definir una meta
def contar_fichas(*nodos):
    nodo=nodos[0]
    tb=nodo.state
    n=len(tb)
    fichas=0
    for i in range(n):
        for j in range(i+1):
            if tb[i][j]==1:
                fichas+=1
    return fichas

def meta(*nodos):
    st=nodos[0]
    return contar_fichas(st)==1

**4. La Heuristíca**<br>
En el código mostramos dos heurísticas utilizadas para A*. <br>
La primera busca estimar el número de movimientos restantes a partir del número de fichas. Sabemos que cada movimiento elimina una ficha, entonces si hay n fichas requerimos al menos n-1 movimientos para llegar a una sola.<br><br>
La segunda heurística considera fichas aisladas, las que no tienen ficha vecina en las 6 direcciones. Esto penaliza configuraciones que contengan fichas aisladas, y de esta forma dos estados que tengan misma cantidad de fichas recibiran valores diferentes, dependiendo de cuantas de estas fichas aisladas existan.

In [ ]:
def heuristica(*nodos):
    nodo = nodos[0]
    return contar_fichas(nodo) - 1

def hcero(*nodos):
    nodo = nodos[0]
    return 0

Para probar las búsquedas se utiliza el siguiente código:

In [ ]:
bfs=sp.TreeSearch(start, sucesorCS, meta, strategy="bfs")
dfs=sp.TreeSearch(start, sucesorCS, meta, strategy="dfs")
bas0=sp.TreeSearch(start, sucesorCS, meta, strategy="a*", heuristic=hcero)
bas=sp.TreeSearch(start, sucesorCS, meta, strategy="a*", heuristic=heuristica)

In [ ]:
r1=bfs.find()
r2=dfs.find()
r3=bas0.find()
r4=bas.find()

In [ ]:
resultados=pd.DataFrame({"Busqueda":["BFS", "DFS", "A*(h=0)", "A*(h(n))"],
                         "Iteraciones":[bfs.iterations, dfs.iterations, bas0.iterations, bas.iterations] })

In [ ]:
resultados

,Busqueda,Iteraciones
0,BFS,3012
1,DFS,484
2,A*(h=0),3012
3,A*(h(n)),473


# **Estimación del tamaño de búsqueda**

Realizamos una estimación del tamaño que podría alcanzar el árbol de búsqueda, para esto utilizamos el factor de ramificación *b* y la profundidad de la solución *d*.<br><br>
Para nuestro problema consideramos un aproximado de **b = 6**<br>
Porque estando en una posición, tenemos 6 posibles direcciones para hacer un movimiento. Es solo una aproximación ya que se sabe que no todos los estados tendrán 6 movimientos válidos.<br><br>
La profundidad **d=13**<br>
Tenemos 15 posiciones, empezamos con 14 fichas y una posición vacía, con cada movimiento eliminamos una ficha, entonces para pasar de 14 fichas a 1 ficha necesitamos de 13 movimientos. <br><br>
Utilizando la aproximación:


In [ ]:
b = 6
d = 13
estimacion = b ** d
print(f"{b}^{d} = {estimacion}")

6^13 = 13060694016


Esto muestra que nuestro número de nodos crecería hasta mas de 13 mil millones.<br>
Pero esto no representa la cantidad real de nodos que serán explorados, ya que con las funciones implementadas solo generamos movimientos válidos y también se evitan expandir de nuevo los estados repetidos, lo cual hará que esta cantidad disminuya.<br><br>
Para nuestro tablero con 15 posiciones obtuvimos lo siguiente:

In [ ]:
resultados

,Busqueda,Iteraciones
0,BFS,3012
1,DFS,484
2,A*(h=0),3012
3,A*(h(n)),473


Las iteraciones son los nodos que fueron expandidos por cada búsqueda, y estos resultados nos ayudan a ver la diferencia entre nuestra estimación y lo que realmente ocurrió. Como se esperaba, podemos ver que hay una gran diferencia que se debe a que nuestra implementación considera solo movimientos válidos y controla a los estados repetidos y, en el calculo **b^d** no se tomaban en cuenta, es únicamente un crecimiento aproximado del árbol de la búsqueda.

# **Protocolo experimental**

**Instancia fácil:** Tablero con 10 agujeros y 9 fichas (solución a 8 movimientos)

1<br>
0 1<br>
1 1 1<br>
1 1 1 1

In [ ]:
#INSTANCIA FÁCIL
startf=sp.node(((1,),(0,1),(1,1,1),(1,1,1,1)), depth=0, parent=None)

fbfs=sp.TreeSearch(startf, sucesorCS, meta, strategy="bfs")
fdfs=sp.TreeSearch(startf, sucesorCS, meta, strategy="dfs")
fbas0=sp.TreeSearch(startf, sucesorCS, meta, strategy="a*", heuristic=hcero)
fbas=sp.TreeSearch(startf, sucesorCS, meta, strategy="a*", heuristic=heuristica)

t1=time.time()
fr1=fbfs.find(max_iter=5000)
fn1=fbfs.iterations
ft1=time.time()-t1

t2=time.time()
fr2=fdfs.find(max_iter=5000)
fn2=fdfs.iterations
ft2=time.time()-t2

t3=time.time()
fr3=fbas0.find(max_iter=5000)
fn3=fbas0.iterations
ft3=time.time()-t3

t4=time.time()
fr4=fbas.find(max_iter=5000)
fn4=fbas.iterations
ft4=time.time()-t4

**Instancia media:** Tablero con 15 agujeros y 14 fichas (solución a 13 movimientos)

1<br>
1 1<br>
1 1 1<br>
1 1 0 1<br>
1 1 1 1 1

In [ ]:
#INSTANCIA MEDIA
startm=sp.node(((1,),(1,1),(1,1,1),(1,1,0,1),(1,1,1,1,1)), depth=0, parent=None)

mbfs=sp.TreeSearch(startm, sucesorCS, meta, strategy="bfs")
mdfs=sp.TreeSearch(startm, sucesorCS, meta, strategy="dfs")
mbas0=sp.TreeSearch(startm, sucesorCS, meta, strategy="a*", heuristic=hcero)
mbas=sp.TreeSearch(startm, sucesorCS, meta, strategy="a*", heuristic=heuristica)

t1=time.time()
mr1=mbfs.find(max_iter=5000)
mn1=mbfs.iterations
mt1=time.time()-t1

t2=time.time()
mr2=mdfs.find(max_iter=5000)
mn2=mdfs.iterations
mt2=time.time()-t2

t3=time.time()
mr3=mbas0.find(max_iter=5000)
mn3=mbas0.iterations
mt3=time.time()-t3

t4=time.time()
mr4=mbas.find(max_iter=5000)
mn4=mbas.iterations
mt4=time.time()-t4

**Instancia superior:** Tablero con 15 agujeros y 14 fichas (solución a 13 movimientos)

1<br>
1 1<br>
1 1 0<br>
1 1 1 1<br>
1 1 1 1 1

In [ ]:
#INSTANCIA SUPERIOR
starts=sp.node(((1,),(1,1),(1,1,0),(1,1,1,1),(1,1,1,1,1)), depth=0, parent=None)

sbfs=sp.TreeSearch(starts, sucesorCS, meta, strategy="bfs")
sdfs=sp.TreeSearch(starts, sucesorCS, meta, strategy="dfs")
sbas0=sp.TreeSearch(starts, sucesorCS, meta, strategy="a*", heuristic=hcero)
sbas=sp.TreeSearch(starts, sucesorCS, meta, strategy="a*", heuristic=heuristica)

t1=time.time()
sr1=sbfs.find(max_iter=5000)
sn1=sbfs.iterations
st1=time.time()-t1

t2=time.time()
sr2=sdfs.find(max_iter=5000)
sn2=sdfs.iterations
st2=time.time()-t2

t3=time.time()
sr3=sbas0.find(max_iter=5000)
sn3=sbas0.iterations
st3=time.time()-t3

t4=time.time()
sr4=sbas.find(max_iter=5000)
sn4=sbas.iterations
st4=time.time()-t4

In [ ]:
pruebas=pd.DataFrame({"Instancia":["Fácil", "Fácil", "Fácil", "Fácil", "Media", "Media", "Media", "Media", "Superior", "Superior", "Superior", "Superior"],
                      "Estrategia":["BFS", "DFS", "A*(h=0)", "A*(h(n))", "BFS", "DFS", "A*(h=0)", "A*(h(n))", "BFS", "DFS", "A*(h=0)", "A*(h(n))"],
                      "Nodos expandidos":[fn1, fn2, fn3, fn4, mn1, mn2, mn3, mn4, sn1, sn2, sn3, sn4],
                      "Tiempo (s)":[ft1, ft2, ft3, ft4, mt1, mt2, mt3, mt4, st1, st2, st3, st4]})

In [ ]:
pruebas

,Instancia,Estrategia,Nodos expandidos,Tiempo (s)
0,Fácil,BFS,61,0.000795
1,Fácil,DFS,36,0.000916
2,Fácil,A*(h=0),61,0.000888
3,Fácil,A*(h(n)),45,0.001205
4,Media,BFS,1650,0.037738
5,Media,DFS,554,0.010405
6,Media,A*(h=0),1650,0.034548
7,Media,A*(h(n)),178,0.004066
8,Superior,BFS,4232,0.101735
9,Superior,DFS,119,0.001808


> Para las mediciones se hizo uso del backend de Google Compute Engine, que utiliza Python 3.

> La RAM del sistema es de 12.67 GB y el Disco es de 107.72 GB


**¿Por qué razón no se incluyen las columnas de longitud y costo?**

En el estado inicial de este problema se tiene un tablero con 15 hoyos y 14 fichas, donde cada movimiento válido corresponderá a la eliminación de una de estas. Dado que, para alcanzar la meta, es necesario dejar únicamente una ficha en el tablero, la solución siempre tendrá una longitud de 13 movimientos. Debido a que se trabaja con un costo unitario por salto, el costo final de la búsqueda será igual a la longitud.

> Para cada estado intermedio, la cantidad de movimientos restantes siempre será igual al número de fichas restantes en el tablero - 1.



# **Análisis de la heurística**

HEURÍSTICA PRINCIPAL:

La heurística implementada se define algebraicamente como:

                        h(n) = F(n) - 1

Donde F(n) es el número de fichas activas presente en el tablero del nodo "n"

Esta función mide la cantidad mínima de saltos requeridos para reducir las fichas actuales hasta la condición de victoria (1 sola ficha).  

Puesto que cada movimiento legal de tipo "salto" elimina exactamente una ficha del tablero, si un estado contiene "f" fichas, se necesitarán al menos "f –1 " saltos para alcanzar el estado meta. Por lo tanto, el número de fichas restante menos uno proporciona un límite inferior natural e intuitivo del esfuerzo restante.  

* DEMOSTRACIÓN DE ADMISIBILIDAD MEDIANTE EL MODELO DE RELAJACIÓN:

Una heurística h(n) es admisible si jamás sobreestima el costo real h(n) para alcanzar la meta, es decir, h(n) <= h*(n) para todo estado b

Para demostrar que nuestra heurística es admisible, recurrimos al método de relajación de problemas. Supongamos un problema "relajado" donde se eliminan por completo las restricciones geométricas del tablero triangular: se ignora si las casillas vecinas están ocupadas o libres, si las fichas están alineadas o si existen límites físicos en la matriz. En este entorno simplificado, el costo exacto para pasar de F(n) fichas a 1 ficha es precisamente F(n) - 1 acciones.

Dado que el problema real no puede resolverse en menos pasos que el problema relajado (ya que en el problema real hay restricciones adicionales que pueden obligar a hacer rodeos o generar callejones sin salido), tenemos que

                    h(n) = F – 1 <= h*(n)  

Donde h*(n) es el costo real mínimo desde el estado n hasta la meta

* ANÁLISIS DEL COMPORTAMIENTO PRÁCTICO Y DEGENERACIÓN DE A*:

A pesar de su solvencia teórica y admisibilidad, la heurística principal presenta un inconveniente práctico severo al integrarse en el algoritmo A*.

Dado que el costo acumulado g(n) a profundidad "d" coincide con el número de movimientos realizados (g(n) = 14 – F), al calcular la función de evaluación de A*



                    f(n) = g(n) + h_1(n)

            f(n) = (14 - F(n)) + (F(n) - 1) = 13



La función para todos los nodos generados en el árbol de búsqueda, f(n) tiene el valor constante de 13. Esto provoca que A* con la heurística no priorice ninguna rama sobre otra y expanda los nodos en orden FIFO, comportándose de manera idéntica a una Búsqueda en Anchura (BFS), o Costo Uniforme (h=0)

SEGUNDA HEURÍSTICA H2: PENALIZACIÓN TOPOLÓGICA POR FICHAS AISLADAS

* DEFINICIÓN Y JUSTIFICACIÓN TOPOLÓGICA:

Para superar la limitación de la primera heurística (que no logra guiar a A* por tener f(n) constante), proponemos una segunda heurística h2 basada en la topología del tablero y el aislamiento de fichas  

h2(n) = (contar fichas(n) -1) + 2 * fichas aisladas(n)

Donde una ficha se considera aislada si no tiene ninguna otra ficha adyacente en ninguna de sus 6 direcciones de salto en el tablero triangular

Esta medida es razonable porque una ficha aislada no puede ejecutar un salto ni servir de puente para que otra la salte. Para "rescatar" o consumir esa ficha, el algoritmo se ve obligado a realizar secuencias complejas de aproximación que consumen movimientos extra. Al agregar un factor de penalización de +2, la heurística le asigna un costo estimado significativamente más alto a aquellos nodos que degradan la cohesión del tablero.

* ¿Es Admisible?: No, h2 es estrictamente no admisible. Al sumar una penalización arbitraria por aislamiento, es posible que para ciertos estados h2(n) > h^*(n), sobreestimando el costo real hacia la meta.  

Una heurística no admisible no está prohibida, pero como ya sabemos que todas las soluciones miden 13 pasos, no nos interesa mantener la admisibilidad para encontrar la ruta óptima, preferimos una heurística agresiva que descarte de inmediato las jugadas malas con fichas aisladas

In [ ]:
def contar_fichas_aisladas(nodo):
    tb = nodo.state
    n = len(tb)
    aisladas = 0

    for i in range(n):
        for j in range(i + 1):
            if tb[i][j] == 1:
                tiene_vecino = False


                if j >= 1 and tb[i][j-1] == 1: #vecino a la izq
                    tiene_vecino = True
                elif j < i and tb[i][j+1] == 1: #vecino a la der
                    tiene_vecino = True
                elif i >= 1 and j >= 1 and tb[i-1][j-1] == 1: #vecino a la izqsup
                    tiene_vecino = True
                elif i >= 1 and j < i and tb[i-1][j] == 1: #vecino a la dersup
                    tiene_vecino = True
                elif i < n - 1 and tb[i+1][j] == 1: #vecino a la izqinf
                    tiene_vecino = True
                elif i < n - 1 and tb[i+1][j+1] == 1: #vecino a la derinf
                    tiene_vecino = True

                if not tiene_vecino:
                    aisladas += 1

    return aisladas

def heuristica_2(*nodos):
    nodo = nodos[0]
    fichas = contar_fichas(nodo)
    aisladas = contar_fichas_aisladas(nodo)

    return (fichas - 1) + (2 * aisladas)

# **Conclusión**

Tras realizar las pruebas con las distintas instancias propuestas, se pudo observar que, en algunos casos, la estrategia que obtuvo mejores resultados fue DFS, mientras que en otros fue A*. Es importante destacar que la cantidad de nodos expandidos obtenida para estas dos estrategias puede variar dependiendo de la posición inicial del hueco en el tablero y también del orden en el que se analizan los movimientos válidos en la función sucesor.

La heurística principal propuesta es admisible, ya que no sobreestima el número de movimientos necesarios para alcanzar un solución. Sin embargo, puede ocurrir que A* no logre priorizar una rama sobre otras.

 Por lo tanto, como una posible mejora del resultado obtenido con A*, sería conveniente encontrar una heurística alternativa que logre ser admisible y que sea capaz de proporcionar estimaciones más positivas, logrando así dar un resultado menor de nodos expandidos.

# **Referencias Bibliográficas**



**Solanki, U. (2026, 19 de junio).** From algorithms to AI foundations. Medium. https://medium.com/@uttkarsh2003.solanki/from-algorithms-to-ai-foundations-4c1c321bb3ff

**Bell, G. I. (2007).** *Solving Triangular Peg Solitaire*. arXiv:math/0703865 [math.CO].
   Recuperado de: https://arxiv.org/abs/math/0703865

**Ortiz Bejar, J. (2026, 6 de septiembre).** NReinas, Github. https://github.com/kyriox/baile/blob/master/doc/NReinas.ipynb




# **Aportaciones de cada integrante**

**Aníbal Zavala Herrera:** Mi contribución al proyecto abarcó tanto la fundamentación teórica como el desarrollo práctico. Participé en la investigación integral del problema, analizando desde la mecánica del juego y sus reglas hasta los diversos algoritmos de búsqueda. En la etapa de programación, trabajé de la mano con el equipo en la estructuración e implementación de cada sección del código base. Mi mayor enfoque se centró en la investigación, propuesta y diseño formal de las funciones heurísticas, garantizando su correcta integración y respaldo teórico.


**Alejandra Oropeza:** Participé junto a mis compañeros en el análisis del problema y en el planteamiento de los cuatros componentes base de la búsqueda (estado, sucesor, meta y heurística), tanto a nivel teórico como durante el desarrollo del código. Realicé algunas propuestas para la función sucesor, para posteriormente seleccionar en conjunto la opción que mejor se ajustara a la mecánica y los requerimientos del juego. Mi aportación se enfocó principalmente en el planteamiento de distintas instancias para el protocolo experimental, así como en el análisis del número de nodos expandidos y del tiempo de cada una de las estrategias de búsqueda empleadas.


**Jorge Luis Pineda Guzmán:** Mi aportación en este proyecto fue, con mis compañeros, durante la elección del problema, poder entenderlo teoricamente y analizar todos los componentes necesarios para continuar con la implementación en python. Durante esta implementación, trabajamos juntos en ver cómo se estructuraría nuestro código desde la base, definiendo los 4 componentes, resolviendo errores y casos inválidos del problema. En el reporte mi aportación fue más hacia la redacción de la validación de los componentes y el por qué cumplen y son correctos para nuestro caso, además de realizar la estimación del tamaño de búsqueda desde un principio y luego comprobarla con los resultados reales, y así entender porque nuestra estimación fue errónea y a qué se debe la diferencia que obtuvimos.     
